# Solar Eclipse Data Transformation

This notebook creates the derived columns required for the dashboard filters:
`Year` and `Eclipse Category`.


## Load dataset

In [ ]:
from pathlib import Path

import pandas as pd

solar_df = pd.read_csv("../backend/data/solar.csv")

solar_df.head()

,Catalog Number,Calendar Date,Eclipse Time,Delta T (s),Lunation Number,Saros Number,Eclipse Type,Gamma,Eclipse Magnitude,Latitude,Longitude,Sun Altitude,Sun Azimuth,Path Width (km),Central Duration
0,1,-1999 June 12,03:14:51,46438,-49456,5,T,-0.2701,1.0733,6.0N,33.3W,74,344,247,06m37s
1,2,-1999 December 5,23:45:23,46426,-49450,10,A,-0.2317,0.9382,32.9S,10.8E,76,21,236,06m44s
2,3,-1998 June 1,18:09:16,46415,-49444,15,T,0.4994,1.0284,46.2N,83.4E,60,151,111,02m15s
3,4,-1998 November 25,05:57:03,46403,-49438,20,A,-0.9045,0.9806,67.8S,143.8W,25,74,162,01m14s
4,5,-1997 April 22,13:19:56,46393,-49433,-13,P,-1.4670,0.1611,60.6S,106.4W,0,281,NaN,NaN


## Create `Year` column

Extract the year from `Calendar Date` so it can be used as a dashboard filter.

- LLM Help about the `-` and reason to keep it. (I dont know anything about timelines in eclipse.)
- Backend: keep Year = -1999 because it sorts and filters correctly.
- Streamlit presentation layer: display it as 2000 BCE, without dash
- Keep the numeric Year column unchanged with the `-` and apply the user-friendly label in the frontend.
  
```
| Stored value | Dashboard label |
| -----------: | --------------- |
|      `-1999` | `2000 BCE`      |
|          `0` | `1 BCE`         |
|       `2010` | `2010 CE`       |
```


In [ ]:
# Extract the year from the Calendar Date column
solar_df["Year"] = (
    solar_df["Calendar Date"]
    .str.split()   # Split each date into separate parts
    .str[0]        # Select the first part, which contains the year
    .astype(int)   # Convert the year from text to an integer
)

# Display the original date and extracted year
solar_df[["Calendar Date", "Year"]].head()

,Calendar Date,Year
0,-1999 June 12,-1999
1,-1999 December 5,-1999
2,-1998 June 1,-1998
3,-1998 November 25,-1998
4,-1997 April 22,-1997


## Create Eclipse Category
Group the detailed NASA eclipse codes into four user-friendly categories.

In [11]:
# Map each eclipse type code to a category name
category_mapping = {
    "P" : "Partial",
    "A" : "Annular",
    "T" : "Total",
    "H" : "Hybrid",
}

# Create a new column containing new eclipse category
solar_df["Eclipse Category"] = (
    solar_df["Eclipse Type"]        # from the OG data
    .str[0]                         # extract 1st letter of the eclipse type
    .map(category_mapping)          # replace letter with full category name from category mapping dict
)

# display both columns
solar_df[["Eclipse Type", "Eclipse Category"]].head()

,Eclipse Type,Eclipse Category
0,T,Total
1,A,Annular
2,T,Total
3,A,Annular
4,P,Partial


In [ ]:
# display each Eclipse Category with total counts each
# total of 11,898 rows

solar_df["Eclipse Category"].value_counts()

Eclipse Category
Partial    4200
Annular    3956
Total      3173
Hybrid      569
Name: count, dtype: int64

## Test the filtering logic for the dashboard

In [20]:
# Test by choosing a year and eclipse category
# Should return the correct value
selected_year = 2010
selected_category = "Annular"


# Filter the DataFrame using the selected year and category
filtered_df = solar_df[
    (solar_df["Year"] == selected_year)
    & (solar_df["Eclipse Category"] == selected_category)
]


# Count the number of matching solar eclipses with len funciton
len(filtered_df)

1

In [ ]:
# To see which eclipses occurred in 2010


solar_df[solar_df["Year"] == 2010][
    ["Calendar Date", "Eclipse Type", "Eclipse Category"]
]

,Calendar Date,Eclipse Type,Eclipse Category
9528,2010 January 15,A,Annular
9529,2010 July 11,T,Total


#### Transformation conclusion

Two derived columns were created for the dashboard:

- `Year`, extracted from `Calendar Date`
- `Eclipse Category`, grouping the detailed eclipse codes into Partial,
  Annular, Total, and Hybrid

The filtering logic was tested successfully using the year 2010 and the
Annular category.